# 🔬 RedPitayaSTCL: Laser Locking Workflow — **Single RP (Cav + Mon)**

**Scanning Transfer Cavity Lock (STCL) using RedPitaya STEMlab 125-14**

This notebook uses **one RedPitaya for both cavity scanning and monitoring** (`mode="scan_mon"`).
No separate Mon board is needed. The single RP scans the cavity via Out2, acquires IN1 as the
cavity signal, and opens the live monitor window — all from the same board.

---

### Board roles

| Label | Mode | Role | Outputs |
|-------|------|------|---------|
| `Cav` | `scan_mon` | Scans cavity **and** monitors signal in real time | OUT2 → ramp, OUT1 → trigger |
| `Lock1` | `lock` | Applies PID feedback to laser current modulation inputs | OUT1 → Slave1, OUT2 → Slave2 |

> **Wiring:** `Cav OUT1` (square-wave trigger) → `IN2` of Lock1.  
> `Cav OUT2` (triangle ramp) → piezo amplifier → cavity piezo.  
> Cavity transmission photodiode → `IN1` of Cav (and optionally Lock1).

---

### Quick reference: scan timing

| `dec` | Sample rate | Buffer duration |
|-------|-------------|----------------|
| 8 | 15.6 MHz | 1.049 ms |
| 16 | 7.8 MHz | 2.097 ms |
| 32 | 3.9 MHz | 4.194 ms |
| 64 | 1.95 MHz | 8.389 ms |

Range and lockpoint values are always in **milliseconds**.

### Monitor aesthetics
The live cavity monitor uses a **dark Catppuccin theme**:
- Dual-axis layout: **IN1 cavity** (top, 3× height) + **IN2 trigger** (bottom) when `SHOW_TRIGGER = True`
- Coloured range spans (pale blue = Master, warm colours = Slaves)
- Amber dashed lockpoint lines
- Toggle `SHOW_TRIGGER` in Phase 0 before starting the monitor

---
## Phase 0: Configuration

<blockquote style="border-left:4px solid #e67e22; padding:6px 12px;
  background:#fdf6ec; color:#7f4f00; border-radius:4px;">
  <strong> → Edit the cells in this phase only </strong> before running the notebook.
  All parameters are passed through to the relevant phases automatically.
</blockquote>

### Board IPs
Set the IP for the Cav board. Comment out `Lock1` if absent.

### Monitor trigger
`SHOW_TRIGGER = True` gives a dual-panel plot (cavity + trigger).
`SHOW_TRIGGER = False` gives a single-panel cavity-only plot.

### Lock parameters
**Cavity range format:** `[[r1_start, r1_end], [r2_start, r2_end]]` in ms.  
**Slave range format:** `[start, end]` in ms.

In [ ]:
# ── Board addresses ────────────────────────────────────────────────────────────
# Cav is a single RP running in scan_mon mode (scans cavity + runs live monitor).
RP_CAV_IP   = "192.168.0.99"   # Cav  — scan_mon RP  (required)
# RP_LOCK1_IP = "192.168.0.102" # Lock1 — laser lock RP (comment out if absent)

SSH_USER = "root"
SSH_PASS = "root"

In [ ]:
# ── Decimation ─────────────────────────────────────────────────────────────────
# Power of 2 between 1 and 65536. Higher = slower scan, more averaging.
# Common values:  dec=8  → 1.049 ms | dec=16 → 2.097 ms | dec=32 → 4.194 ms
CAV_DEC = 32

In [ ]:
# ── Scan signal defaults ────────────────────────────────────────────────────
# LV mode constraint: CAV_AMP + abs(CAV_OFFSET) <= 1.0 V
CAV_AMP    = 0.7   # V  — half-swing of triangle wave
CAV_OFFSET = 0.0   # V  — DC offset shifting the scan centre

_CAV_PERIOD_MS = 8e-9 * 16384 * CAV_DEC * 1e3   # ms, computed from CAV_DEC
print("Scan defaults:  amp={} V  offset={} V  dec={}  period={:.3f} ms".format(
      CAV_AMP, CAV_OFFSET, CAV_DEC, _CAV_PERIOD_MS))

In [ ]:
# ── Monitor trigger visibility ────────────────────────────────────────────────
# True  → dual-axis: IN1 cavity (top, 3×) + IN2 trigger (bottom)
# False → single-axis: IN1 cavity only
# Set BEFORE starting the monitor. To change while running: stop → edit → restart.
SHOW_TRIGGER = True

In [ ]:
# ── Cavity (Master) lock parameters ───────────────────────────────────────────
# Two ranges: one per reference peak. Values in ms.
CAV_RANGE     = [[0.15, 0.50], [1.70, 2.00]]
CAV_LOCKPOINT = 1.80   # ms — target position of the first reference peak
CAV_PID       = {"P": 0.0, "I": 2.0, "D": 0.0,
                 "I_val": 0, "limit": [-0.99, 0.99]}

In [ ]:
# ── Slave 1 lock parameters (Lock1 OUT1) ──────────────────────────────────────
SL1_LABEL     = "Laser_A"
SL1_RANGE     = [0.85, 1.10]   # ms
SL1_LOCKPOINT = 0.96           # ms
SL1_ENABLED   = True
SL1_PID       = {"P": 0.0, "I": 0.5, "D": 0.0,
                 "I_val": 0, "limit": [-0.99, 0.99]}

In [ ]:
# ── Slave 2 lock parameters (Lock1 OUT2) ──────────────────────────────────────
SL2_LABEL     = "Laser_B"
SL2_RANGE     = [0.50, 0.85]   # ms
SL2_LOCKPOINT = 0.60           # ms
SL2_ENABLED   = False          # set True when second laser is coupled in
SL2_PID       = {"P": 0.0, "I": 0.5, "D": 0.0,
                 "I_val": 0, "limit": [-0.99, 0.99]}

In [ ]:
print("Configuration loaded (single RP — scan_mon mode).")
print("  Cav  (scan_mon):", RP_CAV_IP)
# print("  Lock1 (lock)   :", RP_LOCK1_IP)
print(f"  Monitor trigger: {'ON  — dual-axis' if SHOW_TRIGGER else 'OFF — cavity only'}")

---
## Phase 1: Upload & Connect

**What this does:**
1. Adds the repo root to `sys.path` so all imports resolve.
2. Registers `Cav` as `mode="scan_mon"` — the `LockClient` puts it in both
   `self.masters` (for scanning/locking) **and** `self.monitors` (for the live display).
3. SSHes into Cav (and Lock1 if present), uploads RP-side scripts, loads settings.
4. Starts the PC-side selector event loop in a background thread.
5. Applies the `SHOW_TRIGGER` flag to `Monitor` class before any monitor starts.

> **If this cell hangs beyond ~45 s:** SSH in manually and run:
> `PYTHONPATH=/opt/redpitaya/lib/python/:$PYTHONPATH python3 /root/RunLock.py`

In [ ]:
import sys, pathlib, threading, time

# ── Locate repo root and add to path ──────────────────────────────────────────
_here = pathlib.Path().resolve()
for _p in [_here] + list(_here.parents)[:4]:
    if (_p / "lockclient.py").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        print("Repo root:", _p)
        break

from lockclient import LockClient, RP_client, Monitor

In [ ]:
# ── Apply monitor trigger flag ─────────────────────────────────────────────────
# This must run before start_monitor(). The flag is snapshotted per-process
# in Monitor.__init__, so changing it after the process starts has no effect.
Monitor.show_trigger = SHOW_TRIGGER
status = "ON  — dual-axis (IN1 cavity top, IN2 trigger bottom)" if SHOW_TRIGGER else "OFF — single-axis (cavity only)"
print(f"Trigger visibility: {status}")

In [ ]:
# ── Build RP_client dictionary ────────────────────────────────────────────────
#
# 'Cav' uses mode="scan_mon":
#   • LockClient.__init__ adds it to self.masters  → drives the scan loop
#   • LockClient.__init__ adds it to self.monitors → enables start_monitor()
#   No separate Mon board is needed.
#
# The RP-side RunLock.py sees mode="scan" (scan_mon is a PC-side concept only).
# The board script is started with mode="scan" as before.

RPs = {
    "Cav"  : RP_client((RP_CAV_IP,   5000), {}, mode="scan_mon"),
    # "Lock1": RP_client((RP_LOCK1_IP, 5000), {}, mode="lock"),
}

print("Uploading scripts and loading settings...")
Lock = LockClient(RPs)
print("Done.")
print()
print("Boards registered:")
for name, rp in Lock.RPs.items():
    print("  {:6s}  mode={:10s}  addr={}".format(name, rp.mode, rp.addr[0]))
print()
print("Masters  :", Lock.masters)
print("Monitors :", list(Lock.monitors.keys()))

In [ ]:
# ── Connect boards (starts RunLock.py via SSH) ────────────────────────────────
def _wrap(fn, err):
    try:
        fn()
    except Exception as exc:
        err["exc"] = exc

def run_with_timeout(fn, timeout_s, name):
    err = {}
    t = threading.Thread(target=lambda: _wrap(fn, err), daemon=True)
    t.start()
    t.join(timeout=timeout_s)
    if t.is_alive():
        raise TimeoutError(
            "{} timed out after {}s.\n"
            "Troubleshooting:\n"
            "  1. SSH into board and run: PYTHONPATH=/opt/redpitaya/lib/python/:$PYTHONPATH python3 /root/RunLock.py\n"
            "  2. Check port 5000 is free: ss -tlnp | grep 5000\n"
            "  3. Kill stale processes:    pkill -f RunLock.py\n"
            "  4. Power-cycle the board if nothing else works.".format(name, timeout_s)
        )
    if "exc" in err:
        raise RuntimeError("{} failed: {}".format(name, err["exc"]))

run_with_timeout(Lock.connect_all, timeout_s=45, name="connect_all")
print("All boards connected.")

In [ ]:
# ── Start PC-side event loop ──────────────────────────────────────────────────
if "stcl_thread" not in globals() or not stcl_thread.is_alive():
    stcl_thread = threading.Thread(target=Lock.start, daemon=True)
    stcl_thread.start()
    time.sleep(2)
    print("Event loop started.")
else:
    print("Event loop already running.")

# ── Apply CAV_DEC ───────────────────────────────────────────────────────────
Lock.set_dec("Cav", CAV_DEC)
print("Decimation set to", CAV_DEC, "— period ≈ {:.3f} ms".format(8e-9 * 16384 * CAV_DEC * 1e3))

# ── Connection status ──────────────────────────────────────────────────────────
for name, rp in Lock.RPs.items():
    status = "connected" if rp.connected else "DISCONNECTED"
    print("  {:6s}  {}  {}".format(name, rp.addr[0], status))

---
## Phase 2: Signal Verification

Verify the analog path before starting the cavity scan.

**What to check:**
- CH1 (IN1): cavity transmission photodiode signal — flat baseline expected (scan not yet running).
- CH2 (IN2): trigger square wave from `Cav OUT1` looped back to `Cav IN2` (optional wiring check).

**Wiring reminder:**
- `Cav OUT2` → piezo amplifier → cavity piezo
- `Cav OUT1` → `IN2` of Lock1 (trigger sync)
- Cavity transmission photodiode → `Cav IN1`

---
## Phase 3: Cavity Scan

**Goal:** Start the cavity scan on the Cav RP, push cavity settings, and open
the live monitor window — all from the same board.

**Sequence:**
1. Push Master (cavity) settings.
2. Start the scan loop (fires Out2 triangle ramp continuously).
3. Start the live monitor window.
4. Observe peaks, adjust `CAV_RANGE` / `CAV_LOCKPOINT` in Phase 0, then push updates.

> **Stop the scan loop** (`Lock.stop_loop("Cav")`) before starting the cavity lock (Phase 4).
> Never start a lock on top of a running scan loop.

In [ ]:
# ── Push cavity settings ───────────────────────────────────────────────────────
Lock.update_setting("Cav", "Master", "range",      CAV_RANGE)
Lock.update_setting("Cav", "Master", "lockpoint",  CAV_LOCKPOINT)
Lock.update_setting("Cav", "Master", "enabled",    True)
Lock.update_setting("Cav", "Master", "PID",        CAV_PID)

print("Cavity settings pushed:")
print("  range     :", CAV_RANGE, "ms")
print("  lockpoint :", CAV_LOCKPOINT, "ms")
print("  dec       :", CAV_DEC)
print("  PID       :", CAV_PID)


In [ ]:
# ── Start cavity scan loop ─────────────────────────────────────────────────────
Lock.start_scan("Cav", amplitude=CAV_AMP, offset=CAV_OFFSET)
print("Cavity scan started on Cav ({}).".format(RP_CAV_IP))
print("  Out2: triangle  amp={} V  offset={} V  dec={}  period={:.3f} ms".format(
        CAV_AMP, CAV_OFFSET, CAV_DEC, _CAV_PERIOD_MS))

In [ ]:
# ── Adjust scan output while running ──────────────────────────────────────────
# Edit and re-run to update amplitude/offset without stopping the scan.
# LV mode: CAV_AMP + abs(CAV_OFFSET) <= 1.0 V
Lock.set_scan_output("Cav", amplitude=CAV_AMP, offset=CAV_OFFSET)

---
## Phase 3b: Live Cavity Monitor

Start the live monitor window on the Cav board.

> **Important:** The scan loop (`start_scan`) must already be running before
> calling `start_monitor`. The monitor acquires from the same RP that is scanning.

**Trigger visibility** is controlled by `SHOW_TRIGGER` set in Phase 0:
- `True`  → dual-panel: IN1 cavity (top) + IN2 trigger (bottom)
- `False` → single-panel: IN1 cavity only

In [ ]:
# ── Start live cavity monitor on Cav ──────────────────────────────────────────
# The scan loop must be running before calling this.
# Monitor.show_trigger was already applied in Phase 1.

Lock.start_monitor("Cav")
print("Cavity monitor started on Cav.")
print("You should see IN1 updating in the Qt window.")
print()
print("To stop: Lock.stop_monitor('Cav')")

In [ ]:
# ── ① Edit values to update ──────────────────────────────────────────────────
# Change any values below, then run cell ② to push them to the monitor.
# All values in milliseconds.

CAV_DEC       = 32
CAV_RANGE     = [[0.15, 0.30], [1.80, 2.0]]
CAV_LOCKPOINT = 1.90

_CAV_PERIOD_MS = 8e-9 * 16384 * CAV_DEC * 1e3
print(f"Values ready to push:")
print(f"  dec       : {CAV_DEC}   period ≈ {_CAV_PERIOD_MS:.3f} ms")
print(f"  range     : {CAV_RANGE}")
print(f"  lockpoint : {CAV_LOCKPOINT} ms")
print()
print("Run the next cell (②) to apply.")

In [ ]:
# ── ② Push settings to running monitor ───────────────────────────────────────
# Reads CAV_DEC / CAV_RANGE / CAV_LOCKPOINT set in cell ① above.

# 1. Apply decimation on Cav (rescales time axis)
Lock.set_dec("Cav", CAV_DEC)
_CAV_PERIOD_MS = 8e-9 * 16384 * CAV_DEC * 1e3

# 2. Write new values to settings
Lock.update_setting("Cav", "Master", "range",     CAV_RANGE)
Lock.update_setting("Cav", "Master", "lockpoint", CAV_LOCKPOINT)

# 3. Push to running monitor via its queue (set_monitor() handles this automatically
#    via the @_apply_to_monitor decorator on update_setting, but we confirm here)
print(f"Settings pushed to monitor.")
print(f"  dec       : {CAV_DEC}   period ≈ {_CAV_PERIOD_MS:.3f} ms")
print(f"  range     : {CAV_RANGE}")
print(f"  lockpoint : {CAV_LOCKPOINT} ms")

In [ ]:
# ── ③ Stop monitor ────────────────────────────────────────────────────────────
Lock.stop_monitor("Cav")
print("Monitor stopped. Qt window will close.")

In [ ]:
# ── Restart monitor ───────────────────────────────────────────────────────────
# Run after stopping monitor or changing SHOW_TRIGGER.
# If you changed SHOW_TRIGGER, re-run Phase 0 Cell 5 and Phase 1 Cell 11 first.
Lock.start_monitor("Cav")
print("Monitor restarted.")

---
## Phase 4: Cavity Lock

Lock the cavity length by stabilising the reference laser peak position via the
Cav RP Out2 ramp offset.

**Sequence:**
1. Stop the scan loop (`Lock.stop_loop("Cav")`).
2. Start the cavity lock loop (`Lock.start_lock("Cav")`).
3. Verify lock by watching the reference peak position in the monitor.

> **Note:** While the cavity lock is running you **cannot** run the scan loop
> simultaneously — they share the same port (5065). Start the monitor
> *after* the lock loop is running.

In [ ]:
# ── Stop scan loop before starting lock ───────────────────────────────────────
if Lock.RPs["Cav"].loop_running:
    Lock.stop_loop("Cav")
    time.sleep(0.5)
    print("Scan loop stopped. Ready to start cavity lock.")
else:
    print("No scan loop running.")

In [ ]:
# ── Push cavity settings (refresh before lock) ────────────────────────────────
Lock.update_setting("Cav", "Master", "range",      CAV_RANGE)
Lock.update_setting("Cav", "Master", "lockpoint",  CAV_LOCKPOINT)
Lock.update_setting("Cav", "Master", "enabled",    True)
Lock.update_setting("Cav", "Master", "PID",        CAV_PID)
print("Cavity settings refreshed.")

In [ ]:
# ── Start cavity lock ─────────────────────────────────────────────────────────
Lock.start_lock("Cav")
print("Cavity lock started on Cav.")
print("Stop with: Lock.stop_loop('Cav')")

---
## Phase 5: Laser Lock

Lock individual laser frequencies to the cavity resonances.

**Requires:** `Lock1` board (mode `"lock"`). Add it to the `RPs` dict in Phase 1 if absent.

**Sequence:**
1. Push Slave settings.
2. Start the laser lock (`Lock.start_lock("Lock1")`).
3. Monitor errors in Phase 6.

In [ ]:
# ── Check Lock1 is available ───────────────────────────────────────────────────
if "Lock1" not in Lock.RPs:
    print("Lock1 not in RPs dict — skipping laser lock phase.")
    print("Add Lock1 to the RPs dict in Phase 1 and restart from Phase 1.")
else:
    print("Lock1 found:", Lock.RPs["Lock1"].addr[0])
    print("Connected  :", Lock.RPs["Lock1"].connected)

In [ ]:
# ── Push Slave1 settings ───────────────────────────────────────────────────────
if "Lock1" in Lock.RPs:
    Lock.update_setting("Lock1", "Slave1", "label",     SL1_LABEL)
    Lock.update_setting("Lock1", "Slave1", "range",     SL1_RANGE)
    Lock.update_setting("Lock1", "Slave1", "lockpoint", SL1_LOCKPOINT)
    Lock.update_setting("Lock1", "Slave1", "enabled",   SL1_ENABLED)
    Lock.update_setting("Lock1", "Slave1", "PID",       SL1_PID)
    print("Slave1 ({}) settings pushed:".format(SL1_LABEL))
    print("  range     :", SL1_RANGE, "ms")
    print("  lockpoint :", SL1_LOCKPOINT, "ms")
    print("  enabled   :", SL1_ENABLED)

In [ ]:
# ── Push Slave2 settings (only if SL2_ENABLED = True) ─────────────────────────
if "Lock1" in Lock.RPs:
    Lock.update_setting("Lock1", "Slave2", "label",     SL2_LABEL)
    Lock.update_setting("Lock1", "Slave2", "range",     SL2_RANGE)
    Lock.update_setting("Lock1", "Slave2", "lockpoint", SL2_LOCKPOINT)
    Lock.update_setting("Lock1", "Slave2", "enabled",   SL2_ENABLED)
    Lock.update_setting("Lock1", "Slave2", "PID",       SL2_PID)
    print("Slave2 ({}) settings pushed  (enabled={})".format(SL2_LABEL, SL2_ENABLED))

In [ ]:
# ── Start laser lock ───────────────────────────────────────────────────────────
if "Lock1" in Lock.RPs:
    Lock.start_lock("Lock1")
    print("Laser lock started on Lock1.")
    print("Stop with: Lock.stop_loop('Lock1')")

---
## Phase 6: Error Monitor

Opens a live matplotlib window showing the **frequency error** (MHz) of each
laser channel over time. Runs on the Cav board (scan_mon monitors errors).

> Stop the cavity monitor first if it is running — they share the same monitor
> process slot.

In [ ]:
# ── Start cavity signal monitor (Cav board) ───────────────────────────────────
# Useful for watching the cavity signal while the lock is running.
Lock.start_monitor("Cav")
print("Cavity monitor started on Cav.")

In [ ]:
# ── Start error monitor (Cav board) ───────────────────────────────────────────
# tmin: minimum time between error samples in seconds (default 10 ms).

Lock.stop_monitor("Cav")          # stop cavity monitor first if running
time.sleep(0.5)
Lock.start_error_monitor("Cav", tmin=20e-3)
print("Error monitor started. Close the plot window or run the next cell to stop.")

In [ ]:
# ── Save error data ────────────────────────────────────────────────────────────
# import time as _t
# filename = "lock_errors_{}".format(int(_t.time()))
# Lock.monitors["Cav"]["queue_err"].put(("save", filename))
# print("Saving errors to {}.json".format(filename))

---
## Phase 7: Safe Shutdown

**Always shut down in this order:**
1. Stop error / cavity monitors.
2. Stop laser lock loops (Lock1) first.
3. Stop cavity scan/lock on Cav last — it generates the trigger for Lock1.
4. Disconnect all boards.
5. Stop the PC-side event loop.

> `Lock.close()` performs all steps above in the correct order automatically.

In [ ]:
# ── Stop monitors ──────────────────────────────────────────────────────────────
Lock.stop_monitor("Cav")
time.sleep(0.5)
print("Monitor stopped.")

In [ ]:
# ── Stop laser lock ────────────────────────────────────────────────────────────
if "Lock1" in Lock.RPs and Lock.RPs["Lock1"].loop_running:
    Lock.stop_loop("Lock1")
    time.sleep(0.5)
    print("Laser lock stopped.")

In [ ]:
# ── Stop cavity scan / lock ────────────────────────────────────────────────────
if Lock.RPs["Cav"].loop_running:
    Lock.stop_loop("Cav")
    time.sleep(0.5)
    print("Cav scan/lock stopped.")

In [ ]:
# ── Full clean shutdown ────────────────────────────────────────────────────────
Lock.close()
print("All boards disconnected. Session closed.")
print()
print("Board port status after shutdown:")
cmd = 'ss -tlnp | grep -E "5000|5065"'
print(f"  Verify with:  ssh root@<IP> '{cmd}'")